# Irradiancia en Torreón: dos formas de parametrizar en Google Colab

Cuaderno de prueba para el taller *Del dato climático al kWh* (6to Coloquio de Energía, IER-UNAM).

Compara dos maneras de dar controles a un cálculo sin instalar nada:

- **Opción A · Formularios de Colab** (`# @param`): Colab dibuja deslizadores y campos al lado de la celda. No requiere ninguna biblioteca; fuera de Colab la celda es Python normal.
- **Opción B · ipywidgets** (`interact`): funciona en Colab, en Jupyter y en VS Code. La figura se vuelve a dibujar sola al mover un control.

Abrir en Colab: `https://colab.research.google.com/github/AltamarMx/6toColoquioEnergia/blob/main/notebooks/demo_colab_irradiancia.ipynb`


In [ ]:
# Colab ya trae pandas, numpy, matplotlib e ipywidgets; solo falta pvlib.
try:
    import pvlib
except ImportError:
    %pip install -q pvlib


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pvlib.location import Location
from pvlib import irradiance

torreon = Location(latitude=25.54, longitude=-103.41,
                   tz="America/Mexico_City", altitude=1120, name="Torreón")


def calcular(fecha="2026-03-21", dias=1, turbidez=3.0, inclinacion=90, azimut=90, albedo=0.20):
    """Irradiancia de cielo despejado y sobre una superficie inclinada, cada 10 min."""
    fin = pd.Timestamp(fecha) + pd.Timedelta(days=dias)
    horas = pd.date_range(fecha, fin, freq="10min", tz=torreon.tz, inclusive="left")
    cielo = torreon.get_clearsky(horas, model="ineichen", linke_turbidity=turbidez)
    sol = torreon.get_solarposition(horas)
    poa = irradiance.get_total_irradiance(
        surface_tilt=inclinacion, surface_azimuth=azimut,
        solar_zenith=sol["apparent_zenith"], solar_azimuth=sol["azimuth"],
        dni=cielo["dni"], ghi=cielo["ghi"], dhi=cielo["dhi"],
        albedo=albedo, model="isotropic")
    return cielo, poa


def graficar(fecha="2026-03-21", dias=1, turbidez=3.0, inclinacion=90, azimut=90, albedo=0.20):
    cielo, poa = calcular(fecha, dias, turbidez, inclinacion, azimut, albedo)
    dt_h = 10 / 60
    E_horiz = cielo["ghi"].sum() * dt_h / 1000 / dias
    E_poa = poa["poa_global"].sum() * dt_h / 1000 / dias
    fig, ax = plt.subplots(figsize=(9, 3.6))
    cielo["ghi"].plot(ax=ax, color="gray", ls="--", lw=1, label=f"GHI horizontal · {E_horiz:.1f} kWh/m²/día")
    poa["poa_global"].plot(ax=ax, lw=2.5, label=f"Total en la superficie · {E_poa:.1f} kWh/m²/día")
    poa["poa_direct"].plot(ax=ax, lw=1.2, label="Directa")
    poa["poa_sky_diffuse"].plot(ax=ax, lw=1.2, label="Difusa del cielo")
    poa["poa_ground_diffuse"].plot(ax=ax, lw=1.2, label="Reflejada del suelo")
    ax.legend(frameon=False, fontsize=9)
    ax.set_ylabel("Irradiancia (W/m²)"); ax.set_xlabel("Hora local"); ax.grid(alpha=.3)
    ax.set_title(f"{torreon.name} · β = {inclinacion}°, γ = {azimut}°, ρ = {albedo:.2f}, T_L = {turbidez:.1f}")
    plt.tight_layout()
    plt.show()


## Opción A · Formulario de Colab

En Colab, cada comentario `# @param` se convierte en un control junto a la celda. Con `{ run: "auto" }` en el título, la celda se vuelve a ejecutar al mover cualquier control. Fuera de Colab la celda simplemente usa los valores escritos.


In [ ]:
# @title Parámetros del cálculo { run: "auto" }
fecha = "2026-03-21"  # @param {type:"date"}
dias = 1  # @param {type:"slider", min:1, max:14, step:1}
turbidez = 3.0  # @param {type:"slider", min:2, max:6, step:0.1}
inclinacion = 90  # @param {type:"slider", min:0, max:90, step:5}
azimut = 90  # @param {type:"slider", min:0, max:355, step:5}
albedo = 0.2  # @param {type:"slider", min:0, max:1, step:0.05}

graficar(fecha, dias, turbidez, inclinacion, azimut, albedo)


## Opción B · ipywidgets

`interact` toma una función y un control por argumento. Funciona igual en Colab, Jupyter y VS Code.


In [ ]:
from ipywidgets import interact, IntSlider, FloatSlider, Text

interact(graficar,
         fecha=Text("2026-03-21", description="fecha"),
         dias=IntSlider(1, 1, 14, description="días"),
         turbidez=FloatSlider(3.0, min=2, max=6, step=0.1, description="turbidez"),
         inclinacion=IntSlider(90, 0, 90, 5, description="β (°)"),
         azimut=IntSlider(90, 0, 355, 5, description="γ (°)"),
         albedo=FloatSlider(0.2, min=0, max=1, step=0.05, description="albedo"));
